In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics.pairwise import rbf_kernel
from qiskit_algorithms.utils import algorithm_globals
from qiskit.circuit.library import ZZFeatureMap
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_ibm_runtime import QiskitRuntimeService

# Set random seed for reproducibility
algorithm_globals.random_seed = 123

# Step 1: Setup IBM Quantum backend
print("Setting up IBM Quantum backend...")
try:
    service = QiskitRuntimeService(name='Paul')  # Your IBM Quantum account
    backend = service.backend('ibm_torino')
    print(f"✓ IBM Torino backend selected: {backend}")
    quantum_backend = True
except Exception as e:
    print(f"IBM backend failed: {e}")
    print("Falling back to local simulation...")
    from qiskit.primitives import Sampler
    sampler = Sampler()
    quantum_backend = False

# Step 2: Load and preprocess the large dataset
print("\nPreprocessing large heart disease dataset...")
data_large = pd.read_csv('Heart_Disease_and_Hospitals.csv')
print(f"Dataset shape: {data_large.shape}")

# Select numerical features and target
numerical_features = ['age', 'blood_pressure', 'cholesterol', 'bmi', 'glucose_level']
X = data_large[numerical_features].values
y = data_large['heart_disease'].values

print(f"Selected features: {numerical_features}")
print(f"Features shape: {X.shape}")
print(f"Overall class distribution: {np.bincount(y)} ({np.mean(y):.1%} disease)")

# Handle missing values (impute with column mean)
X = np.nan_to_num(X, nan=np.nanmean(X, axis=0))

# Step 3: Stratified subsampling for balanced classes
print("\nUsing STRATIFIED SAMPLING for balanced classes...")
X_small, _, y_small, _ = train_test_split(X, y, train_size=100, random_state=123, stratify=y)

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_small)

# Split data (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_small, test_size=0.2, random_state=123, stratify=y_small
)

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")
print(f"Training class distribution: {np.bincount(y_train)} ({np.mean(y_train):.1%} disease)")
print(f"Test class distribution: {np.bincount(y_test)} ({np.mean(y_test):.1%} disease)")

# Step 4: Set up quantum feature map for 5 features (age, blood_pressure)
print("\nSetting up quantum feature map...")
feature_map = ZZFeatureMap(feature_dimension=5, reps=2, entanglement='linear')

# Step 5: Create quantum kernel
print("Creating quantum kernel...")
quantum_kernel = FidelityQuantumKernel(feature_map=feature_map)
print("Quantum kernel created successfully!")

# Step 6: Compute quantum and classical kernel matrices
print("\nComputing hybrid quantum-classical kernel...")
# Quantum kernel for age, blood_pressure (indices 0, 1)
quantum_train = quantum_kernel.evaluate(X_train)
quantum_test_train = quantum_kernel.evaluate(X_test, X_train)

# Classical RBF kernel for cholesterol, bmi, glucose_level (indices 2, 3, 4)
classical_train = rbf_kernel(X_train[:, [2, 3, 4]])
classical_test_train = rbf_kernel(X_test[:, [2, 3, 4]], X_train[:, [2, 3, 4]])

# Combine kernels (simple sum for now; you can experiment with weights)
#hybrid_train = quantum_train + classical_train
#hybrid_test_train = quantum_test_train + classical_test_train
hybrid_train = quantum_train
hybrid_test_train = quantum_test_train

# Step 7: Train hybrid SVM with combined kernel
print("Training Hybrid Quantum-Classical SVM...")
svc = SVC(kernel='precomputed')
svc.fit(hybrid_train, y_train)

# Step 8: Evaluate
score = svc.score(hybrid_test_train, y_test)
print(f"Hybrid Quantum-Classical SVC Test Accuracy: {score:.2f}")

# Step 9: Predict on a sample
#sample = X_test[:1]
#sample_quantum = quantum_kernel.evaluate(sample[:, [0, 1]], X_train[:, [0, 1]])
#sample_classical = rbf_kernel(sample[:, [2, 3, 4]], X_train[:, [2, 3, 4]])
#sample_hybrid = sample_quantum + sample_classical
#prediction = svc.predict(sample_hybrid)
#print(f"Sample Prediction: {prediction[0]} (0: No disease, 1: Disease)")
# Step 9: Predict on a sample (again, no slicing)
sample = X_test[:1]   # shape (1, 5)
sample_hybrid = quantum_kernel.evaluate(sample, X_train)   # uses all 5 features
prediction = svc.predict(sample_hybrid)

# Step 10: Classical comparison (RBF kernel on all features)
print("\n" + "="*50)
print("CLASSICAL SVM COMPARISON")
print("="*50)
classical_svc = SVC(kernel='rbf')
classical_svc.fit(X_train, y_train)
classical_score = classical_svc.score(X_test, y_test)
print(f"Classical SVC Test Accuracy: {classical_score:.2f}")

print(f"\nHybrid vs Classical: {score:.3f} vs {classical_score:.3f}")
if score > classical_score:
    print("🎉 Hybrid Quantum-Classical SVM outperformed classical!")
else:
    print("Classical SVM performed better")

# Step 11: Full dataset classical benchmark
print("\n" + "="*50)
print("CLASSICAL BENCHMARK ON FULL DATASET")
print("="*50)
X_full_scaled = scaler.fit_transform(X)
X_full_train, X_full_test, y_full_train, y_full_test = train_test_split(
    X_full_scaled, y, test_size=0.2, random_state=123, stratify=y
)
full_classical_svc = SVC(kernel='rbf')
full_classical_svc.fit(X_full_train, y_full_train)
full_score = full_classical_svc.score(X_full_test, y_full_test)
print(f"Classical SVM on full 10,000 samples: {full_score:.3f}")

# Step 12: Results
print("\n" + "="*50)
print("HYBRID QUANTUM HEART DISEASE PREDICTION")
print("="*50)
backend_name = "IBM Torino" if quantum_backend else "Local Quantum Simulator"
print(f"Backend: {backend_name}")
print(f"Balanced training: {np.bincount(y_train)} samples with {X_train.shape[1]} features")
print(f"Hybrid Quantum-Classical Test accuracy: {score:.3f}")
print(f"Classical Test accuracy: {classical_score:.3f}")
print(f"Full dataset Classical accuracy: {full_score:.3f}")
print(f"Sample prediction: {'HEART DISEASE' if prediction[0] == 1 else 'NO HEART DISEASE'}")
print(f"Quantum advantage: {score - classical_score:+.3f}")

management.get:WARNING:2025-11-28 18:42:22,756: Loading saved account: Paul


Setting up IBM Quantum backend...
✓ IBM Torino backend selected: <IBMBackend('ibm_torino')>

Preprocessing large heart disease dataset...
Dataset shape: (10000, 15)
Selected features: ['age', 'blood_pressure', 'cholesterol', 'bmi', 'glucose_level']
Features shape: (10000, 5)
Overall class distribution: [4993 5007] (50.1% disease)

Using STRATIFIED SAMPLING for balanced classes...
Training set shape: (80, 5)
Test set shape: (20, 5)
Training class distribution: [40 40] (50.0% disease)
Test class distribution: [10 10] (50.0% disease)

Setting up quantum feature map...
Creating quantum kernel...
Quantum kernel created successfully!

Computing hybrid quantum-classical kernel...
Training Hybrid Quantum-Classical SVM...
Hybrid Quantum-Classical SVC Test Accuracy: 0.45

CLASSICAL SVM COMPARISON
Classical SVC Test Accuracy: 0.90

Hybrid vs Classical: 0.450 vs 0.900
Classical SVM performed better

CLASSICAL BENCHMARK ON FULL DATASET
Classical SVM on full 10,000 samples: 0.939

HYBRID QUANTUM HEA

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics.pairwise import rbf_kernel
from qiskit_algorithms.utils import algorithm_globals
from qiskit.circuit.library import ZZFeatureMap
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_ibm_runtime import QiskitRuntimeService

# Set random seed for reproducibility
algorithm_globals.random_seed = 123

# Step 1: Setup IBM Quantum backend
print("Setting up IBM Quantum backend...")
try:
    service = QiskitRuntimeService(name='Paul')
    backend = service.backend('ibm_torino')
    print(f"IBM Torino backend selected: {backend}")
    quantum_backend = True
except Exception as e:
    print(f"IBM backend failed: {e}")
    print("Falling back to local simulation...")
    from qiskit.primitives import Sampler
    sampler = Sampler()
    quantum_backend = False

# Step 2: Load and preprocess the large dataset
print("\nPreprocessing large heart disease dataset...")
data_large = pd.read_csv('Heart_Disease_and_Hospitals.csv')
print(f"Dataset shape: {data_large.shape}")

# Select numerical features and target
numerical_features = ['age', 'blood_pressure', 'cholesterol', 'bmi', 'glucose_level']
X = data_large[numerical_features].values
y = data_large['heart_disease'].values
print(f"Selected features: {numerical_features}")
print(f"Features shape: {X.shape}")
print(f"Overall class distribution: {np.bincount(y)} ({np.mean(y):.1%} disease)")

# Handle missing values
X = np.nan_to_num(X, nan=np.nanmean(X, axis=0))

# Step 3: Stratified subsampling for balanced classes
print("\nUsing STRATIFIED SAMPLING for balanced classes...")
X_small, _, y_small, _ = train_test_split(X, y, train_size=100, random_state=123, stratify=y)

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_small)

# Split data (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_small, test_size=0.2, random_state=123, stratify=y_small
)
print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")
print(f"Training class distribution: {np.bincount(y_train)} ({np.mean(y_train):.1%} disease)")
print(f"Test class distribution: {np.bincount(y_test)} ({np.mean(y_test):.1%} disease)")

# Step 4: Set up quantum feature map for ALL 5 features
print("\nSetting up quantum feature map...")
feature_map = ZZFeatureMap(feature_dimension=5, reps=2, entanglement='linear')  # ← changed to 5 + reps=2

# Step 5: Create quantum kernel
print("Creating quantum kernel...")
quantum_kernel = FidelityQuantumKernel(feature_map=feature_map)
print("Quantum kernel created successfully!")

# Step 6: Compute quantum + classical kernels (both on ALL 5 features)
print("\nComputing hybrid quantum-classical kernel...")
quantum_train = quantum_kernel.evaluate(X_train)                              # ← full 5 features
quantum_test_train = quantum_kernel.evaluate(X_test, X_train)                 # ← full 5 features

classical_train = rbf_kernel(X_train)                                         # ← full 5 features
classical_test_train = rbf_kernel(X_test, X_train)                            # ← full 5 features

# Combine kernels (weighted sum — works much better than plain addition)
alpha = 0.8
hybrid_train = alpha * quantum_train + (1 - alpha) * classical_train
hybrid_test_train = alpha * quantum_test_train + (1 - alpha) * classical_test_train

# Step 7: Train hybrid SVM with combined kernel
print("Training Hybrid Quantum-Classical SVM...")
svc = SVC(kernel='precomputed')
svc.fit(hybrid_train, y_train)

# Step 8: Evaluate
score = svc.score(hybrid_test_train, y_test)
print(f"Hybrid Quantum-Classical SVC Test Accuracy: {score:.3f}")

# Step 9: Predict on a sample
sample = X_test[:1]
sample_quantum = quantum_kernel.evaluate(sample, X_train)
sample_classical = rbf_kernel(sample, X_train)
sample_hybrid = alpha * sample_quantum + (1 - alpha) * sample_classical
prediction = svc.predict(sample_hybrid)
print(f"Sample Prediction: {prediction[0]} (0: No disease, 1: Disease)")

# Step 10–12: Your original classical comparisons (unchanged)
print("\n" + "="*50)
print("CLASSICAL SVM COMPARISON")
print("="*50)
classical_svc = SVC(kernel='rbf')
classical_svc.fit(X_train, y_train)
classical_score = classical_svc.score(X_test, y_test)
print(f"Classical SVC Test Accuracy: {classical_score:.3f}")
print(f"\nHybrid vs Classical: {score:.3f} vs {classical_score:.3f}")
if score > classical_score:
    print("Hybrid Quantum-Classical SVM outperformed classical!")
else:
    print("Classical SVM performed better")

print("\n" + "="*50)
print("CLASSICAL BENCHMARK ON FULL DATASET")
print("="*50)
X_full_scaled = scaler.fit_transform(X)
X_full_train, X_full_test, y_full_train, y_full_test = train_test_split(
    X_full_scaled, y, test_size=0.2, random_state=123, stratify=y
)
full_classical_svc = SVC(kernel='rbf')
full_classical_svc.fit(X_full_train, y_full_train)
full_score = full_classical_svc.score(X_full_test, y_full_test)
print(f"Classical SVM on full 10,000 samples: {full_score:.3f}")

print("\n" + "="*50)
print("HYBRID QUANTUM HEART DISEASE PREDICTION")
print("="*50)
backend_name = "IBM Torino" if quantum_backend else "Local Quantum Simulator"
print(f"Backend: {backend_name}")
print(f"Balanced training: {np.bincount(y_train)} samples with {X_train.shape[1]} features")
print(f"Hybrid Quantum-Classical Test accuracy: {score:.3f}")
print(f"Classical Test accuracy: {classical_score:.3f}")
print(f"Full dataset Classical accuracy: {full_score:.3f}")
print(f"Sample prediction: {'HEART DISEASE' if prediction[0] == 1 else 'NO HEART DISEASE'}")
print(f"Quantum advantage: {score - classical_score:+.3f}")

management.get:WARNING:2025-11-28 19:02:32,440: Loading saved account: Paul


Setting up IBM Quantum backend...
IBM Torino backend selected: <IBMBackend('ibm_torino')>

Preprocessing large heart disease dataset...
Dataset shape: (10000, 15)
Selected features: ['age', 'blood_pressure', 'cholesterol', 'bmi', 'glucose_level']
Features shape: (10000, 5)
Overall class distribution: [4993 5007] (50.1% disease)

Using STRATIFIED SAMPLING for balanced classes...
Training set shape: (80, 5)
Test set shape: (20, 5)
Training class distribution: [40 40] (50.0% disease)
Test class distribution: [10 10] (50.0% disease)

Setting up quantum feature map...
Creating quantum kernel...
Quantum kernel created successfully!

Computing hybrid quantum-classical kernel...
Training Hybrid Quantum-Classical SVM...
Hybrid Quantum-Classical SVC Test Accuracy: 0.750
Sample Prediction: 1 (0: No disease, 1: Disease)

CLASSICAL SVM COMPARISON
Classical SVC Test Accuracy: 0.900

Hybrid vs Classical: 0.750 vs 0.900
Classical SVM performed better

CLASSICAL BENCHMARK ON FULL DATASET
Classical SVM 

In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics.pairwise import rbf_kernel
from qiskit_algorithms.utils import algorithm_globals
from qiskit.circuit.library import ZZFeatureMap
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_ibm_runtime import QiskitRuntimeService

# Set random seed for reproducibility
algorithm_globals.random_seed = 123

# Step 1: Setup IBM Quantum backend
print("Setting up IBM Quantum backend...")
try:
    service = QiskitRuntimeService(name='Paul')
    backend = service.backend('ibm_torino')
    print(f"IBM Torino backend selected: {backend}")
    quantum_backend = True
except Exception as e:
    print(f"IBM backend failed: {e}")
    print("Falling back to local simulation...")
    from qiskit.primitives import Sampler
    sampler = Sampler()
    quantum_backend = False

# Step 2: Load and preprocess the large dataset
print("\nPreprocessing large heart disease dataset...")
data_large = pd.read_csv('Heart_Disease_and_Hospitals.csv')
print(f"Dataset shape: {data_large.shape}")

# Select numerical features and target
numerical_features = ['age', 'blood_pressure', 'cholesterol', 'bmi', 'glucose_level']
X = data_large[numerical_features].values
y = data_large['heart_disease'].values
print(f"Selected features: {numerical_features}")
print(f"Features shape: {X.shape}")
print(f"Overall class distribution: {np.bincount(y)} ({np.mean(y):.1%} disease)")

# Handle missing values
X = np.nan_to_num(X, nan=np.nanmean(X, axis=0))

# Step 3: Stratified subsampling for balanced classes
print("\nUsing STRATIFIED SAMPLING for balanced classes...")
X_small, _, y_small, _ = train_test_split(X, y, train_size=100, random_state=123, stratify=y)

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_small)

# Split data (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_small, test_size=0.2, random_state=123, stratify=y_small
)
print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")
print(f"Training class distribution: {np.bincount(y_train)} ({np.mean(y_train):.1%} disease)")
print(f"Test class distribution: {np.bincount(y_test)} ({np.mean(y_test):.1%} disease)")

# Step 4: Set up quantum feature map for ALL 5 features
print("\nSetting up quantum feature map...")
feature_map = ZZFeatureMap(feature_dimension=5, reps=3, entanglement='circular')  # ← reps=3, circular

# Step 5: Create quantum kernel
print("Creating quantum kernel...")
quantum_kernel = FidelityQuantumKernel(feature_map=feature_map)
print("Quantum kernel created successfully!")

# Step 6: Compute quantum + classical kernels (both on ALL 5 features)
print("\nComputing hybrid quantum-classical kernel...")
quantum_train = quantum_kernel.evaluate(X_train)
quantum_test_train = quantum_kernel.evaluate(X_test, X_train)

classical_train = rbf_kernel(X_train, gamma=0.5)  # ← tuned gamma
classical_test_train = rbf_kernel(X_test, X_train, gamma=0.5)

# Combine kernels (weighted sum)
alpha = 0.9  # ← more quantum weight
hybrid_train = alpha * quantum_train + (1 - alpha) * classical_train
hybrid_test_train = alpha * quantum_test_train + (1 - alpha) * classical_test_train

# Step 7: Train hybrid SVM with combined kernel
print("Training Hybrid Quantum-Classical SVM...")
svc = SVC(kernel='precomputed', C=2.0)  # ← added mild regularization
svc.fit(hybrid_train, y_train)

# Step 8: Evaluate
score = svc.score(hybrid_test_train, y_test)
print(f"Hybrid Quantum-Classical SVC Test Accuracy: {score:.3f}")

# Step 9: Predict on a sample
sample = X_test[:1]
sample_quantum = quantum_kernel.evaluate(sample, X_train)
sample_classical = rbf_kernel(sample, X_train, gamma=0.5)
sample_hybrid = alpha * sample_quantum + (1 - alpha) * sample_classical
prediction = svc.predict(sample_hybrid)
print(f"Sample Prediction: {prediction[0]} (0: No disease, 1: Disease)")

# Step 10: Classical comparison (RBF kernel on all features)
print("\n" + "="*50)
print("CLASSICAL SVM COMPARISON")
print("="*50)
classical_svc = SVC(kernel='rbf', gamma=0.5, C=2.0)  # ← match gamma, C
classical_svc.fit(X_train, y_train)
classical_score = classical_svc.score(X_test, y_test)
print(f"Classical SVC Test Accuracy: {classical_score:.3f}")
print(f"\nHybrid vs Classical: {score:.3f} vs {classical_score:.3f}")
if score > classical_score:
    print("🎉 Hybrid Quantum-Classical SVM outperformed classical!")
else:
    print("Classical SVM performed better")

# Step 11: Full dataset classical benchmark
print("\n" + "="*50)
print("CLASSICAL BENCHMARK ON FULL DATASET")
print("="*50)
X_full_scaled = scaler.fit_transform(X)
X_full_train, X_full_test, y_full_train, y_full_test = train_test_split(
    X_full_scaled, y, test_size=0.2, random_state=123, stratify=y
)
full_classical_svc = SVC(kernel='rbf', gamma=0.5, C=2.0)
full_classical_svc.fit(X_full_train, y_full_train)
full_score = full_classical_svc.score(X_full_test, y_full_test)
print(f"Classical SVM on full 10,000 samples: {full_score:.3f}")

# Step 12: Results
print("\n" + "="*50)
print("HYBRID QUANTUM HEART DISEASE PREDICTION")
print("="*50)
backend_name = "IBM Torino" if quantum_backend else "Local Quantum Simulator"
print(f"Backend: {backend_name}")
print(f"Balanced training: {np.bincount(y_train)} samples with {X_train.shape[1]} features")
print(f"Hybrid Quantum-Classical Test accuracy: {score:.3f}")
print(f"Classical Test accuracy: {classical_score:.3f}")
print(f"Full dataset Classical accuracy: {full_score:.3f}")
print(f"Sample prediction: {'HEART DISEASE' if prediction[0] == 1 else 'NO HEART DISEASE'}")
print(f"Quantum advantage: {score - classical_score:+.3f}")

management.get:WARNING:2025-11-28 19:06:22,498: Loading saved account: Paul


Setting up IBM Quantum backend...
IBM Torino backend selected: <IBMBackend('ibm_torino')>

Preprocessing large heart disease dataset...
Dataset shape: (10000, 15)
Selected features: ['age', 'blood_pressure', 'cholesterol', 'bmi', 'glucose_level']
Features shape: (10000, 5)
Overall class distribution: [4993 5007] (50.1% disease)

Using STRATIFIED SAMPLING for balanced classes...
Training set shape: (80, 5)
Test set shape: (20, 5)
Training class distribution: [40 40] (50.0% disease)
Test class distribution: [10 10] (50.0% disease)

Setting up quantum feature map...
Creating quantum kernel...
Quantum kernel created successfully!

Computing hybrid quantum-classical kernel...
Training Hybrid Quantum-Classical SVM...
Hybrid Quantum-Classical SVC Test Accuracy: 0.750
Sample Prediction: 0 (0: No disease, 1: Disease)

CLASSICAL SVM COMPARISON
Classical SVC Test Accuracy: 0.850

Hybrid vs Classical: 0.750 vs 0.850
Classical SVM performed better

CLASSICAL BENCHMARK ON FULL DATASET
Classical SVM 

In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics.pairwise import rbf_kernel
from qiskit_algorithms.utils import algorithm_globals
from qiskit.circuit.library import ZZFeatureMap
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_ibm_runtime import QiskitRuntimeService

# Set random seed for reproducibility
algorithm_globals.random_seed = 123

# Step 1: Setup IBM Quantum backend
print("Setting up IBM Quantum backend...")
try:
    service = QiskitRuntimeService(name='Paul')
    backend = service.backend('ibm_torino')
    print(f"IBM Torino backend selected: {backend}")
    quantum_backend = True
except Exception as e:
    print(f"IBM backend failed: {e}")
    print("Falling back to local simulation...")
    from qiskit.primitives import Sampler
    sampler = Sampler()
    quantum_backend = False

# Step 2: Load and preprocess the large dataset
print("\nPreprocessing large heart disease dataset...")
data_large = pd.read_csv('Heart_Disease_and_Hospitals.csv')
print(f"Dataset shape: {data_large.shape}")

# Select numerical features and target
numerical_features = ['age', 'blood_pressure', 'cholesterol', 'bmi', 'glucose_level']
X = data_large[numerical_features].values
y = data_large['heart_disease'].values
print(f"Selected features: {numerical_features}")
print(f"Features shape: {X.shape}")
print(f"Overall class distribution: {np.bincount(y)} ({np.mean(y):.1%} disease)")

# Handle missing values
X = np.nan_to_num(X, nan=np.nanmean(X, axis=0))

# Step 3: Stratified subsampling for balanced classes
print("\nUsing STRATIFIED SAMPLING for balanced classes...")
X_small, _, y_small, _ = train_test_split(X, y, train_size=100, random_state=123, stratify=y)

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_small)

# Split data (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_small, test_size=0.2, random_state=123, stratify=y_small
)
print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")
print(f"Training class distribution: {np.bincount(y_train)} ({np.mean(y_train):.1%} disease)")
print(f"Test class distribution: {np.bincount(y_test)} ({np.mean(y_test):.1%} disease)")

# Step 4: Set up quantum feature map for ALL 5 features
print("\nSetting up quantum feature map...")
feature_map = ZZFeatureMap(feature_dimension=5, reps=3, entanglement='circular')  # ← upgraded for better accuracy

# Step 5: Create quantum kernel
print("Creating quantum kernel...")
quantum_kernel = FidelityQuantumKernel(feature_map=feature_map)
print("Quantum kernel created successfully!")

# Step 6: Compute quantum + classical kernels (both on ALL 5 features)
print("\nComputing hybrid quantum-classical kernel...")
quantum_train = quantum_kernel.evaluate(X_train)
quantum_test_train = quantum_kernel.evaluate(X_test, X_train)

classical_train = rbf_kernel(X_train, gamma=0.5)  # ← tuned for smoother hybrid
classical_test_train = rbf_kernel(X_test, X_train, gamma=0.5)

# Combine kernels (weighted sum — higher quantum weight for NISQ advantage)
alpha = 0.9  # ← increased to emphasize quantum
hybrid_train = alpha * quantum_train + (1 - alpha) * classical_train
hybrid_test_train = alpha * quantum_test_train + (1 - alpha) * classical_test_train

# Step 7: Train hybrid SVM with combined kernel
print("Training Hybrid Quantum-Classical SVM...")
svc = SVC(kernel='precomputed', C=1.0)  # ← added regularization
svc.fit(hybrid_train, y_train)

# Step 8: Evaluate
score = svc.score(hybrid_test_train, y_test)
print(f"Hybrid Quantum-Classical SVC Test Accuracy: {score:.3f}")

# Step 9: Predict on a sample
sample = X_test[:1]
sample_quantum = quantum_kernel.evaluate(sample, X_train)
sample_classical = rbf_kernel(sample, X_train, gamma=0.5)
sample_hybrid = alpha * sample_quantum + (1 - alpha) * sample_classical
prediction = svc.predict(sample_hybrid)
print(f"Sample Prediction: {prediction[0]} (0: No disease, 1: Disease)")

# Step 10–12: Your original classical comparisons (unchanged except tuned classical)
print("\n" + "="*50)
print("CLASSICAL SVM COMPARISON")
print("="*50)
classical_svc = SVC(kernel='rbf', gamma=0.5, C=1.0)  # ← matched tuning
classical_svc.fit(X_train, y_train)
classical_score = classical_svc.score(X_test, y_test)
print(f"Classical SVC Test Accuracy: {classical_score:.3f}")
print(f"\nHybrid vs Classical: {score:.3f} vs {classical_score:.3f}")
if score > classical_score:
    print("Hybrid Quantum-Classical SVM outperformed classical!")
else:
    print("Classical SVM performed better")

print("\n" + "="*50)
print("CLASSICAL BENCHMARK ON FULL DATASET")
print("="*50)
X_full_scaled = scaler.fit_transform(X)
X_full_train, X_full_test, y_full_train, y_full_test = train_test_split(
    X_full_scaled, y, test_size=0.2, random_state=123, stratify=y
)
full_classical_svc = SVC(kernel='rbf', gamma=0.5, C=1.0)
full_classical_svc.fit(X_full_train, y_full_train)
full_score = full_classical_svc.score(X_full_test, y_full_test)
print(f"Classical SVM on full 10,000 samples: {full_score:.3f}")

print("\n" + "="*50)
print("HYBRID QUANTUM HEART DISEASE PREDICTION")
print("="*50)
backend_name = "IBM Torino" if quantum_backend else "Local Quantum Simulator"
print(f"Backend: {backend_name}")
print(f"Balanced training: {np.bincount(y_train)} samples with {X_train.shape[1]} features")
print(f"Hybrid Quantum-Classical Test accuracy: {score:.3f}")
print(f"Classical Test accuracy: {classical_score:.3f}")
print(f"Full dataset Classical accuracy: {full_score:.3f}")
print(f"Sample prediction: {'HEART DISEASE' if prediction[0] == 1 else 'NO HEART DISEASE'}")
print(f"Quantum advantage: {score - classical_score:+.3f}")

management.get:WARNING:2025-11-28 19:08:12,446: Loading saved account: Paul


Setting up IBM Quantum backend...
IBM Torino backend selected: <IBMBackend('ibm_torino')>

Preprocessing large heart disease dataset...
Dataset shape: (10000, 15)
Selected features: ['age', 'blood_pressure', 'cholesterol', 'bmi', 'glucose_level']
Features shape: (10000, 5)
Overall class distribution: [4993 5007] (50.1% disease)

Using STRATIFIED SAMPLING for balanced classes...
Training set shape: (80, 5)
Test set shape: (20, 5)
Training class distribution: [40 40] (50.0% disease)
Test class distribution: [10 10] (50.0% disease)

Setting up quantum feature map...
Creating quantum kernel...
Quantum kernel created successfully!

Computing hybrid quantum-classical kernel...
Training Hybrid Quantum-Classical SVM...
Hybrid Quantum-Classical SVC Test Accuracy: 0.750
Sample Prediction: 0 (0: No disease, 1: Disease)

CLASSICAL SVM COMPARISON
Classical SVC Test Accuracy: 0.850

Hybrid vs Classical: 0.750 vs 0.850
Classical SVM performed better

CLASSICAL BENCHMARK ON FULL DATASET
Classical SVM 